# Quranic SQL Basics with DuckDB CLI

Simple SQL queries for exploring the JSON datasets in `quran-data`.

---

## Datasets

- `surah.json`: Surah details and revelation place.
- `ayah.json`: Verse text and word counts.
- `juz.json`: Juz divisions and verse counts.
- `sajda.json`: Prostration verses.
- `matching-ayah.json`: Nested verse similarity matches.

## Run the SQL with DuckDB CLI

Each SQL cell starts DuckDB CLI with `%%script duckdb`. The SQL body is sent directly to the CLI, so the cells can be run from this notebook.

Make sure duckdb CLI is installed
On Windows:

`winget install DuckDB.cli`

On Mac:

`brew install duckdb`

### Question 1: Surahs and Verses by Revelation Place

Count total surahs and total verses for Makkah vs Madinah, including each place's percentage of the total.

**SQL Concepts:** `WITH`, `GROUP BY`, `COUNT`, `SUM`, window functions.

In [21]:
%%script duckdb
-- Count surahs and verses, then calculate each place's percentage of the total.
-- A CTE is a temporary named result used to organize a query.
WITH surah_stats AS (
    SELECT
        revelation_place,
        COUNT(*) AS total_surahs,
        SUM(verses_count) AS total_verses
    FROM 'quran-data/surah.json'
    GROUP BY revelation_place
)
SELECT
    revelation_place,
    total_surahs,
    -- Overall percentage of surahs and verses by revelation place.
    -- SUM(...) OVER () totals all groups while keeping each group as a row.
    ROUND(100.0 * total_surahs / SUM(total_surahs) OVER (), 2) AS surah_percentage,
    total_verses,
    ROUND(100.0 * total_verses / SUM(total_verses) OVER (), 2) AS verse_percentage
FROM surah_stats
ORDER BY total_surahs DESC;

┌──────────────────┬──────────────┬──────────────────┬──────────────┬──────────────────┐
│ revelation_place │ total_surahs │ surah_percentage │ total_verses │ verse_percentage │
│     varchar      │    int64     │      double      │    int128    │      double      │
├──────────────────┼──────────────┼──────────────────┼──────────────┼──────────────────┤
│ Makkah           │           86 │            75.44 │         4613 │            73.97 │
│ Madinah          │           28 │            24.56 │         1623 │            26.03 │
└──────────────────┴──────────────┴──────────────────┴──────────────┴──────────────────┘


### Question 2: Surahs with Prostration Verses (Sajdah)

List Sajdah verses with their Surah names and Arabic text.

**SQL Concepts:** `JOIN` matches related rows; `ON` defines the matching columns; `ORDER BY` sorts the final results.

In [ ]:
%%script duckdb
-- SELECT chooses the columns to display.
SELECT
    s.sajdah_number,
    s.verse_key,
    s.sajdah_type,
    su.name_arabic AS surah_name, -- AS gives the output column a clear name.
    su.name_english AS surah_name_english,
    a.text AS verse_text
FROM 'quran-data/sajda.json' AS s
-- JOIN adds verse details when both verse keys match.
JOIN 'quran-data/ayah.json' AS a
    ON s.verse_key = a.verse_key
-- JOIN adds Surah names when the Surah numbers match.
JOIN 'quran-data/surah.json' AS su
    ON a.surah_number = su.id
-- ORDER BY sorts the final rows by Sajdah number.
ORDER BY s.sajdah_number;

┌───────────────┬───────────┬─────────────┬────────────┬────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ sajdah_number │ verse_key │ sajdah_type │ surah_name │ surah_name_english │                                                                                      verse_text                                                                                       │
│     int64     │  varchar  │   varchar   │  varchar   │      varchar       │                                                                                        varchar                                                                                        │
├───────────────┼───────────┼─────────────┼────────────┼────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

### Question 3: Top 5 Longest Juz by Verse Count

Find the 5 Juz with the most verses, including the names and text of their starting and ending ayahs.

**SQL Concepts:** `SELECT` chooses columns; `split_part` extracts parts of a verse key; `JOIN` combines related rows; `ORDER BY` sorts results; `LIMIT` keeps the top five.

In [ ]:
%%script duckdb
-- SELECT chooses the Juz columns needed for the first step.
SELECT
    juz_number,
    verses_count,
    first_verse_key,
    last_verse_key
FROM 'quran-data/juz.json'
-- ORDER BY DESC puts the largest verse counts first.
ORDER BY verses_count DESC
-- LIMIT returns only the five largest Juz.
LIMIT 5;

┌────────────┬──────────────┬─────────────────┬────────────────┐
│ juz_number │ verses_count │ first_verse_key │ last_verse_key │
│   int64    │    int64     │     varchar     │    varchar     │
├────────────┼──────────────┼─────────────────┼────────────────┤
│         30 │          564 │ 78:1            │ 114:6          │
│         29 │          431 │ 67:1            │ 77:50          │
│         27 │          399 │ 51:31           │ 57:29          │
│         23 │          357 │ 36:28           │ 39:31          │
│         19 │          339 │ 25:21           │ 27:55          │
└────────────┴──────────────┴─────────────────┴────────────────┘


In [7]:
%%script duckdb
-- SELECT returns the Juz boundaries, Surah names, and verse text.
SELECT
    j.juz_number,
    j.verses_count,
    -- split_part gets the Surah and ayah numbers from keys such as '2:255'.
    split_part(j.first_verse_key, ':', 1)::INT AS start_surah,
    start_s.name_arabic AS start_surah_name,
    split_part(j.first_verse_key, ':', 2)::INT AS start_aya,
    start_a.text AS start_aya_text,
    split_part(j.last_verse_key, ':', 1)::INT AS end_surah,
    end_s.name_arabic AS end_surah_name,
    split_part(j.last_verse_key, ':', 2)::INT AS end_aya,
    end_a.text AS end_aya_text
FROM 'quran-data/juz.json' AS j
-- JOIN finds the starting and ending ayahs by their verse keys.
JOIN 'quran-data/ayah.json' AS start_a
    ON j.first_verse_key = start_a.verse_key
JOIN 'quran-data/ayah.json' AS end_a
    ON j.last_verse_key = end_a.verse_key
-- JOIN finds each boundary ayah's Surah name.
JOIN 'quran-data/surah.json' AS start_s
    ON start_a.surah_number = start_s.id
JOIN 'quran-data/surah.json' AS end_s
    ON end_a.surah_number = end_s.id
-- Sort by verse count and keep the five largest Juz.
ORDER BY j.verses_count DESC
LIMIT 5;

-- COPY saves the previous result as a Parquet file for later use.
COPY _ TO 'quran-data/top_juz.parquet' (FORMAT PARQUET);

-- Read the saved Parquet file back as a table.
SELECT * FROM 'quran-data/top_juz.parquet';

┌────────────┬──────────────┬─────────────┬──────────────────┬───────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┬────────────────┬─────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ juz_number │ verses_count │ start_surah │ start_surah_name │ start_aya │                                             start_aya_text                                              │ end_surah │ end_surah_name │ end_aya │                                               end_aya_text                                                │
│   int64    │    int64     │    int32    │     varchar      │   int32   │                                                 varchar                                                 │   int32   │    varchar     │  int32  │                                                  varchar                                                  │
├────────────┼──

### Question 4: Top 5 Surahs by Total Word Count

Find the 5 largest Surahs by total word count.

**SQL Concepts:** `JOIN` matches each Surah to its ayahs; `SUM` adds the words; `GROUP BY ALL` creates one total per Surah; `ORDER BY` and `LIMIT` return the top five.

In [ ]:
%%script duckdb
-- JOIN connects each Surah to its ayahs using the Surah number.
SELECT
    su.name_arabic AS surah_name,
    su.name_english AS surah_name_english,
    su.revelation_place,
    -- SUM adds the word counts for all ayahs in each Surah.
    SUM(a.words_count) AS total_words
FROM 'quran-data/surah.json' AS su
JOIN 'quran-data/ayah.json' AS a
    ON su.id = a.surah_number
-- GROUP BY ALL groups by every non-aggregated selected column.
GROUP BY ALL
-- Sort totals from largest to smallest and keep five rows.
ORDER BY total_words DESC
LIMIT 5;

┌────────────┬────────────────────┬──────────────────┬─────────────┐
│ surah_name │ surah_name_english │ revelation_place │ total_words │
│  varchar   │      varchar       │     varchar      │   int128    │
├────────────┼────────────────────┼──────────────────┼─────────────┤
│ البقرة     │ Al-Baqarah         │ Madinah          │        6117 │
│ النساء     │ An-Nisa            │ Madinah          │        3747 │
│ آل عمران   │ Ali 'Imran         │ Madinah          │        3481 │
│ الأعراف    │ Al-A'raf           │ Makkah           │        3320 │
│ الأنعام    │ Al-An'am           │ Makkah           │        3050 │
└────────────┴────────────────────┴──────────────────┴─────────────┘


### Question 5: Longest Verses

Find the 10 verses with the most words and characters.

**SQL Concepts:** `SELECT` chooses columns; `length` counts characters; `AS` gives a calculated value a name; `ORDER BY` sorts results; `LIMIT` restricts the number of rows.

In [6]:
%%script duckdb
-- SELECT chooses the verse columns to display.
SELECT
    verse_key,
    words_count,
    -- length counts the characters in each verse; AS names the result.
    length(text) AS character_count,
    text
FROM 'quran-data/ayah.json'
-- Sort from the most words to the fewest.
ORDER BY words_count DESC
-- LIMIT keeps the result short and easy to inspect.
LIMIT 10;

┌───────────┬─────────────────┬─────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ verse_key │ character_count │ words_count │                                                                                                                                                                                                                                  

In [ ]:
%%script duckdb
-- COPY saves the result of the query as a reusable Parquet file.
COPY (
    SELECT
        revelation_place,
        -- AVG calculates the average verses per Surah in each group.
        AVG(verses_count) AS avg_verse_count,
        -- COUNT counts how many Surahs are in each group.
        COUNT(*) AS surah_count
    FROM 'quran-data/surah.json'
    -- GROUP BY creates one summary row per revelation place.
    GROUP BY revelation_place
    -- ORDER BY shows the groups with more Surahs first.
    ORDER BY surah_count DESC
) TO 'quran-data/summary_by_revelation_place.parquet' (FORMAT parquet);